# 阶段2：杠杆测算（统一口径）

**目的：** 各模块的建议原本使用不同的单位（延误率、延误占比、超额差评等），无法比较。这里把主要杠杆统一换算成"**分析期内可减少的差评数**"，回答章程的主问题：哪些杠杆的投入产出最高？

**方法：** 静态上限估计（D-008、D-034）。假设其他条件不变，把某个问题改善到基准水平，计算差评会减少多少。各杠杆之间有部分重叠（例如高峰月延误中也包含卖家超时造成的延误），**不能直接相加**。

**数据：** `dws_order_fulfillment`、`ads_m6_seller_tier`；模块3、6 的结论

In [ ]:
from olist_utils import run_query, set_style, COLORS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
set_style()

## 1. 转换系数

把"少延误多少单"换算成"少多少差评"：延误让差评率增加约 53 个百分点；卖家发货超时让延误风险增加约 16 个百分点。

In [ ]:
# 两个"转换系数"：从数据里现算
sql_lift = """
SELECT AVG(CASE WHEN is_late = 1 THEN is_bad_review END)
     - AVG(CASE WHEN is_late = 0 THEN is_bad_review END)                    AS bad_lift_late,   -- 延误让差评率增加多少
       AVG(CASE WHEN is_seller_late = 1 THEN is_late END)
     - AVG(CASE WHEN is_seller_late = 0 THEN is_late END)                   AS late_lift_ship   -- 发货超时让延误风险增加多少
FROM dws_order_fulfillment
"""
lift = run_query(sql_lift).astype(float).iloc[0]
lift

## 2. 各杠杆的测算

In [ ]:
# 杠杆 1：高峰月延误率降到平常月水平（高峰月定义见 D-032）
SPIKE = "('2017-11', '2018-02', '2018-03')"
sql_peak = f"""
SELECT SUM(CASE WHEN purchase_month IN {SPIKE} THEN 1 ELSE 0 END)          AS spike_orders,
       SUM(CASE WHEN purchase_month IN {SPIKE} THEN is_late ELSE 0 END)    AS spike_late,
       AVG(CASE WHEN purchase_month NOT IN {SPIKE} THEN is_late END)       AS normal_late_rate
FROM dws_order_fulfillment
"""
pk = run_query(sql_peak).astype(float).iloc[0]
late_avoided_peak = pk["spike_late"] - pk["spike_orders"] * pk["normal_late_rate"]

# 杠杆 2：RJ 平常月的延误率降到 SP 水平（只算平常月，避免和杠杆 1 重复）
sql_rj = f"""
SELECT SUM(CASE WHEN customer_state = 'RJ' THEN 1 ELSE 0 END)        AS rj_orders,
       AVG(CASE WHEN customer_state = 'RJ' THEN is_late END)          AS rj_late_rate,
       AVG(CASE WHEN customer_state = 'SP' THEN is_late END)          AS sp_late_rate
FROM dws_order_fulfillment
WHERE purchase_month NOT IN {SPIKE}
"""
rj = run_query(sql_rj).astype(float).iloc[0]
late_avoided_rj = rj["rj_orders"] * (rj["rj_late_rate"] - rj["sp_late_rate"])

# 杠杆 3：卖家发货超时全部消除（模块3 3.3、模块6 6.3）
sql_ship = """
SELECT SUM(is_seller_late)                                                  AS ship_late_orders,
       SUM(CASE WHEN is_seller_late = 1 AND is_late = 0 THEN has_review END) AS ship_late_ontime_reviewed
FROM dws_order_fulfillment
WHERE is_seller_late IS NOT NULL
"""
sh = run_query(sql_ship).astype(float).iloc[0]
WITHIN_SELLER_EFFECT = 0.0276      # 模块6 6.3：卖家内部对比得到的等待效应
late_avoided_ship = sh["ship_late_orders"] * lift["late_lift_ship"]
bad_ship_wait = sh["ship_late_ontime_reviewed"] * WITHIN_SELLER_EFFECT

# 杠杆 4：40 个头部体验/双重问题卖家的未延误差评率降到平台平均（模块6 6.2）
sql_sellers = """
SELECT COUNT(*)                                          AS sellers,
       SUM(n_ot_bad - n_ot_reviewed * platform_ot_bad)   AS excess_bad
FROM ads_m6_seller_tier
WHERE is_head = 1 AND problem_type IN ('体验问题', '双重问题')
"""
se = run_query(sql_sellers).astype(float).iloc[0]

# 杠杆 5：多件订单（准时送达）的差评率降到单件订单水平（模块3 3.7）
sql_multi = """
SELECT SUM(CASE WHEN n_items > 1 THEN has_review END)                                   AS multi_reviewed,
       SUM(CASE WHEN n_items > 1 THEN is_bad_review END)                                AS multi_bad,
       SUM(CASE WHEN n_items = 1 THEN is_bad_review END) / SUM(CASE WHEN n_items = 1 THEN has_review END) AS single_rate
FROM dws_order_fulfillment
WHERE is_late = 0
"""
mu = run_query(sql_multi).astype(float).iloc[0]
excess_multi = mu["multi_bad"] - mu["multi_reviewed"] * mu["single_rate"]

# 延误订单只有约 97.7% 有评价：把"少延误的订单数"换算成"少的差评数"时乘上评价率
sql_rev = "SELECT AVG(has_review) AS review_rate FROM dws_order_fulfillment WHERE is_late = 1"
review_rate = float(run_query(sql_rev).iloc[0, 0])
to_bad = review_rate * lift["bad_lift_late"]

total_bad = float(run_query("SELECT SUM(is_bad_review) FROM dws_order_fulfillment").iloc[0, 0])

levers = pd.DataFrame([
    ["高峰月延误率降到平常水平",        late_avoided_peak * to_bad,                 "物流运力预案",       "中"],
    ["发货超时全部消除",                late_avoided_ship * to_bad + bad_ship_wait, "平台规则 + 提醒",    "低"],
    ["40 个头部问题卖家整改到平均水平", se["excess_bad"],                           "卖家运营",           "低"],
    ["多件订单差评率降到单件水平",      excess_multi,                               "需先诊断原因",       "未知"],
    ["RJ 平常月延误率降到 SP 水平",     late_avoided_rj * to_bad,                   "区域承运商",         "高"],
], columns=["杠杆", "可减少差评（分析期内，上限）", "主要手段", "实施难度"])
levers["占全部差评"] = levers["可减少差评（分析期内，上限）"] / total_bad
levers = levers.sort_values("可减少差评（分析期内，上限）", ascending=False).reset_index(drop=True)
print(f"全部差评：{total_bad:,.0f}")
levers.round({"可减少差评（分析期内，上限）": 0, "占全部差评": 3})

## 3. 杠杆对比图

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(levers))[::-1]
ax.barh(y, levers["可减少差评（分析期内，上限）"], color=COLORS["blue"], height=0.6)
for yi, v, s in zip(y, levers["可减少差评（分析期内，上限）"], levers["占全部差评"]):
    ax.text(v + 15, yi, f"{v:,.0f} 条（{s:.1%}）", va="center", fontsize=9.5)
ax.set_yticks(y)
ax.set_yticklabels(levers["杠杆"])
ax.set_xlim(0, levers["可减少差评（分析期内，上限）"].max() * 1.3)
ax.set_xticks([])
ax.grid(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_visible(False)
ax.set_title("最大的两个杠杆是多件订单和高峰月延误，RJ 常态治理收益最小", loc="left")
ax.text(0.0, -0.1, "静态上限估计（D-008、D-034）：假设其他条件不变；各杠杆之间有部分重叠，不能直接相加",
        transform=ax.transAxes, fontsize=8, color=COLORS["text_secondary"])
fig.tight_layout()
fig.savefig("../docs/figures/s2_levers.png", dpi=150, bbox_inches="tight")
plt.show()

**结论：**
- 最大的两个杠杆是**多件订单**（约 1,535 条，但原因尚未确认，需要先诊断）和**高峰月延误**（约 1,155 条）
- **发货超时治理**（约 880 条）和**头部问题卖家整改**（约 860 条）实施难度低，适合作为短期动作
- **RJ 平常月的延误治理收益很小**（约 150 条）：RJ 在全国延误中占比高，主要是高峰月放大的结果，治理重点应放在高峰期的运力预案